# Data Literacy Workshop: Day 1 — Practice Notebook 1
## Full Pipeline Repetition on a New Dataset Variant

### Overview

* This is a **practice repetition** of the full Day 1 workflow, done independently after the live workshop, using a **different synthetic dataset** than the one used in the workshop session (same schema, different fault mix and values — so the answers you memorized won't just transfer, you have to apply the technique again).
* You will work with data coming from multiple systems and formats, and learn how to ingest, inspect, and combine these sources into a consistent analytical dataset.
* The workflow in this notebook emphasizes:
    - Profiling and exploring datasets
    - Assessing data quality and applying validation rules
    - Cleaning and standardizing structured fields
    - Detecting and handling missing values, duplicates, and invalid identifiers
    - Normalizing categorical, numeric, and date attributes
    - Along the way, you will identify common issues in raw data and apply techniques to correct them using Python and pandas.
* By the end of this notebook, you will have transformed messy, inconsistent transactional inputs into a clean, well-structured dataset, and persisted the final output for downstream analytical or machine-learning use.

## What you will do:

In this notebook, you will walk through the complete process of preparing transactional data for analysis. Specifically, you will:
* Load data from multiple sources — CSV files, API-style responses, and Mongo DB.
* Inspect and understand each dataset’s structure, schema, and key fields.
* Profile the data to reveal patterns, distributions, and potential quality issues.
* Detect and handle missing values, placeholders, invalid codes, and inconsistent formatting.
* Identify and remove duplicate records across sources.
* Normalize and standardize categorical labels, currency codes, region codes, and product attributes.
* Validate key fields such as transaction IDs, masked PANs, merchant category codes, and country identifiers.
* Merge and reconcile data from all three systems into a unified, canonical dataset.
* Apply business rules to assess data correctness and flag anomalies.
* Generate summaries and insights that describe data quality and dataset readiness.
* Persist the final cleaned dataset into SQLite for use in analytics, dashboards, or machine-learning workflows.

To execute the code in this notebook, always choose Python Environment, Python 3.8.10 /bin/python Kernel.

## About Dataset

This practice run uses a **second synthetic dataset** generated with a different random seed from the same generator used for the workshop (`DataSet/Generate_Data_Script_WithFault.py`), stored under `DataSet/Practice/`. The schema and card-transaction lifecycle scenario are identical to the workshop notebook — only the values and fault mix differ.

For the Data Literacy Day 1 workshop, you will work with three complementary datasets, each representing a different stage in the card-transaction lifecycle.

1. Settlement Dataset (CSV)
* Represents the Clearing & Settlement systems.
* This dataset serves as the final, authoritative view of each transaction and is used for reconciliation and settlement.
* It includes:
    - Financial attributes
    - Product and card attributes
    - Issuer and Acquirer information
    - Merchant country/region details

2. API Dataset
* Represents Point-of-Authorization (real-time POS) transaction data.
* This reflects what an issuer sees at the exact moment a transaction is authorized.
* It typically includes:
    - Masked PAN
    - POS entry mode & environment
    - Merchant category and merchant name
    - Authorization response code
    - Authorization currency (currency requested at POS)
* In this practice notebook, the API data is accessed through a second mock API server instance (see Exercise 1.2 / the Mock API section below for the practice port).

3. MongoDB Dataset
* Represents enrichment data related to Risk, Tokenization, and ECI/MOTO classification.
* These fields change frequently and are better suited to flexible JSON structures, making MongoDB an appropriate storage system.
* Contains:
    - Risk scores
    - Tokenization indicators
    - ECI/MOTO classification
    - Nested and flexible JSON structures

## Activities Overview

This workshop is organized into the following hands-on Activities:
1. Imports and Configuration
2. Data acquisition 
3. Utility functions
4. Preprocessing & flags
5. Merge & canonicalization
6. Cleaning & canonical fixes
7. Visualizations
8. Persistence and metadata
9. Conclusion

## Activity 1 : Imports and Configuration

In this Activity, you will:
* Set up your environment by importing all the necessary Python libraries for:
    - data manipulation, 
    - visualization and 
    - accessing web and database sources
* Define and configure key variables, such as:
    - File paths for your datasets and outputs, 
    - The address for a mock API, 
    - A list of common placeholder values that represent missing data. 

#### Exercise 1.1 : Import packages 

To get started, we will import all the Python libraries required for data processing, visualization, and accessing external data sources in this notebook.

In [ ]:
from pathlib import Path
from typing import List, Dict, Any, Tuple, Sequence
from datetime import datetime
import json
import requests
import sqlite3
import re
import pandas as pd
import matplotlib.pyplot as plt
import base64, io, html as _html
%matplotlib inline

#### Exercise 1.2 : Paths and Config

Next, we will define and configure the key file paths, API endpoints, and placeholder values needed for data ingestion and processing throughout this notebook.

**Practice-run paths** (different from the workshop notebook, so this run does not overwrite workshop outputs):
* `DATA_DIR` -> `DataSet/Practice`
* `OUTPUT_DIR` -> `ProcessedData/Practice1`

In [ ]:
# TODO: Define the following configuration values:
#   - DATA_DIR: Path to the "DataSet" folder
#   - SETTLEMENT_CSV: DATA_DIR / "settlement.csv"
#   - OUTPUT_DIR: Path to the "ProcessedData" folder; create it if it doesn't already
#     exist (see Path.mkdir(parents=True, exist_ok=True))
#   - PLACEHOLDERS: a set of string values that commonly indicate missing data in this
#     dataset, e.g. {"", " ", "-", "0", "00", "0000", "999", "9999", "NA", "N/A", "None", "nan"}

#### Summary

* You now have a fully configured environment prepared for the rest of the workflow.
* All essential libraries, paths, and configuration settings are in place, giving you a stable foundation for data ingestion, exploration, and analysis in the upcoming activities.

## Activity 2: Data Acquisition 

* In this activity, you will load data from three different sources that represent distinct stages of the transaction lifecycle: 
    * A CSV file, 
    * paginated web API and 
    * A MongoDB database. 
* You will use custom helper functions to read the data robustly, handling potential issues like missing files and API failures. 
* For each source, you will load the data into a pandas DataFrame and perform a quick inspection to understand its shape and see the first few rows.
* This step establishes a clear understanding of the raw inputs you’ll be working with for the rest of the notebook.

#### Helper Exercise: A reusable inspection function
You are about to load three different sources (CSV, API, MongoDB) and will want to check each one the same way: shape, a sample of rows, dtype/null info, and the top values in one column. Rather than retyping that ritual three times, define it once here and reuse it after each load below.

In [ ]:
def inspect_dataframe(df: pd.DataFrame, name: str, sample_col: str = None) -> None:
    """
    Print a quick, consistent inspection summary for a raw source DataFrame: shape, a
    3-row sample, dtype/null info, and (optionally) the top-5 most frequent values in
    one representative column.

    Args:
        df (pd.DataFrame): The DataFrame to inspect.
        name (str): A human-readable label for the source, used in the printed headers.
        sample_col (str, optional): A column to show a top-5 value_counts() for.
            Skipped if not provided or not present in df.
    """
    # Steps:
    #   1. Print "<name> shape:" followed by df.shape
    #   2. display() a 3-row sample: df.head(3)
    #   3. Print "<name> DataFrame Info:" then call df.info()
    #   4. If sample_col is given and is a column in df, print a small header and
    #      display(df[sample_col].value_counts().head(5))
    # TODO: implement
    pass 


### Activity 2.1 : Read CSV Data

* In this step, you will load the CSV file that represents Visa’s clearing and settlement system.
* This dataset serves as the first official, system-of-record view of a transaction and includes details such as issuer and acquirer countries, merchant category codes (MCCs), currency codes, network identifiers, and product attributes.
* Many fields follow Visa-specific coding patterns, such as "840 - US" or "5411 - Grocery Stores".
* CSV files remain one of the most widely used formats for internal data exchange. Financial CSVs commonly include:
    - Placeholder values such as " ", "0", or "-"
    - Embedded metadata in label formats like "NNNN - Description"
    - String-based codes that should not be automatically converted by pandas
* For these reasons, it is best to load all columns as strings initially, ensuring that no unintended type conversions occur during ingestion.

#### Exercise 2.1.1 : Define function to read CSV file into a dataframe

In [ ]:
# Define read_csv_strict function
def read_csv_strict(path: Path) -> pd.DataFrame:
    '''
    Reads a CSV file into a DataFrame with strict type handling.
    Ensures all data is read as strings, trims whitespace, and raises an error if the file is missing.
    Args
        path (Path): Path to the CSV file.
    Returns:
        pd.DataFrame: DataFrame with all data as strings and trimmed whitespace.
    Raises:
        FileNotFoundError: If the specified file does not exist.
    '''
    # Steps:
    #   1. If the file at `path` does not exist, raise FileNotFoundError with a helpful message
    #   2. Otherwise, read it with pd.read_csv using:
    #        dtype=str, keep_default_na=False, na_values=[], low_memory=False
    #      (this stops pandas from guessing column types or silently turning values into NaN)
    #   3. Return the resulting DataFrame
    # TODO: implement
    pass 


#### Exercise 2.1.2 : Load the Settlement Data from the CSV File

In [ ]:
# TODO:
#   1. Print a loading message, e.g. "Loading settlement (CSV)..."
#   2. Call read_csv_strict(SETTLEMENT_CSV) and store the result in df_settlement_raw
#   3. Call inspect_dataframe(df_settlement_raw, "Settlement (Raw)", sample_col="tran_curr_cd")

#### Exercise 2.1.3 : Define the Canonical Column Set

In [ ]:
# Canonical column set 
CANONICAL_COLS = [
    "cpd_mnth_id","tran_id_raw","pymt_crd_acct_num","ntwrk_id","issr_ctry_cd","issr_regn_cd",
    "issr_usr_bus_id","lgl_ntwrk_grp_cd","acqr_bin_ctry_cd","acqr_bin_regn_cd","tran_curr_cd",
    "eci_moto_cd","mrch_catg_cd","mrch_nm_raw","mrch_ctry_cd","mrch_ctry_alpha_cd","mrch_regn_cd",
    "acct_prod_id","prod_brnd_cd","prod_typ_cd","crd_id_mthd_cd","pos_entry_mode_cd","pos_env_cd",
    "pos_terml_cpbty_cd","pin_entry_cpbty_cd","auth_resp_cd"
]

* In this activity, you established a reliable approach for working with the settlement CSV dataset.
* You created a helper function to load CSV files safely, imported the settlement data into a DataFrame, and defined the canonical column set that will guide all future cleaning and validation steps.

### Activity 2.2 : Read API Data

* The API represents the authorization layer — the moment a card is tapped or swiped.
* Sometimes contains fields that settlement doesn’t have (e.g., POS entry mode).
* Real systems rely on pagination, timeouts, fallbacks, and retry logic.
* For this workshop, we will simulate accessing API data using a **mock API server**. This allows us to work with realistic data without needing to connect to a live, external service.

#### Starting the Mock API Server
To ensure the smooth execution of the notebook, please start the **practice** mock API server (a separate instance from the workshop one, serving the practice dataset on a different port) by following these steps:

1.  Open a new Terminal: Navigate to `View-> Terminal`.
2.  Navigate to the `DataSet/Practice` folder:
    ```
      cd DataSet/Practice
    ```
3.  Start the Mock API server on port 8001 (workshop uses 8000; using a different port lets both run at once if needed):
    ```
      uvicorn mock_api:app --reload --host 127.0.0.1 --port 8001
    ```

    Once the server has started, the synthetic API transaction data will be available at:
     `http://127.0.0.1:8001/transactions`

    To continue accessing the data from the API, the mock API server must be in running state. Do not close this terminal if you want to access the data from the API. Just minimise the Terminal area.

4. You can now proceed with fetching the data. In the following exercise, you will use a dedicated API-loading helper function to fetch this authorization data from our mock API and load it into a pandas DataFrame.

#### Exercise 2.2.1 : Define helper function to read the data from API 

* In this exercise, you create a helper function that retrieves records from a paginated API.
* The function loops through pages, collects the returned data, handles basic errors, and finally converts all retrieved rows into a pandas DataFrame.
* This gives you a clean and consistent way to load authorization data from the mock API for further exploration and cleaning.

In [ ]:
# Define fetch_api_paginated function
def fetch_api_paginated(base_url: str, count: int = 000, page_size: int = 10000, timeout=5.0) -> pd.DataFrame:
    """
    Fetch data from a paginated API endpoint.

    Args:
        base_url (str): The base URL of the API endpoint.
        count (int, optional): Total number of records to fetch. If None or <=0, fetch until no more data. Defaults to 000.
        page_size (int, optional): Number of records per page. Defaults to 10000.
        timeout (float, optional): Request timeout in seconds. Defaults to 5.0.

    Returns:
        pd.DataFrame: DataFrame containing the fetched data.
    """
    # Steps:
    #   1. Keep a list `all_rows` to accumulate fetched records, and a `page` counter starting at 1
    #   2. If `count` is None or <= 0, treat it as "fetch until no more data"
    #   3. Loop:
    #        a. Work out how many records to request this page (respecting `count` and `page_size`)
    #        b. GET base_url with params {"count": ..., "page": page}, using `timeout`
    #        c. Raise for HTTP errors, parse the JSON body, and read its "data" list
    #        d. If "data" is empty, stop -- there is no more data
    #        e. Extend `all_rows` with the new rows; stop early once you have `count` rows
    #        f. If the response includes meta.total_pages and you have reached it, stop
    #        g. Otherwise increment `page` and continue
    #        h. On any request exception, print the error and stop the loop
    #   4. Convert all_rows to a DataFrame and return it
    # TODO: implement
    pass


#### Exercise 2.2.2 : Fetch Data from API 

* In this exercise, you use the API-loading helper to fetch authorization data from the mock API and load it into a DataFrame.

In [ ]:
# TODO:
#   1. Print a status message, e.g. "Attempting to fetch API data from mock API..."
#   2. Set API_BASE = "http://127.0.0.1:8000/transactions"
#   3. Call fetch_api_paginated(API_BASE, count=80000, page_size=1000) and store as df_api_raw
#   4. If df_api_raw is empty (mock API not running / no fallback), print a warning and build an
#      empty DataFrame using the CANONICAL_COLS that also exist in df_settlement_raw
#   5. Call inspect_dataframe(df_api_raw, "API (Raw)", sample_col="mrch_catg_cd")

### Activity 2.3 : Read MongoDB Data

* This represents Risk & Enrichment Systems such as fraud engines.
* Contains additional context not present in the core flows. 

#### Exercise 2.3.1 : Helper function to read data from MongoDB 

* In this exercise, you define a helper function to read documents from a MongoDB Atlas collection.
* The function connects to the specified database, retrieves documents based on an optional query and limit, and then normalizes the JSON results into a pandas DataFrame.
* This gives you a straightforward and reusable way to extract enriched transaction data stored in MongoDB.

In [ ]:
#  Define read_mongo_atlas function
def read_mongo_atlas(uri: str, db_name: str, coll_name: str, query: dict = {}, limit: int = 100000) -> pd.DataFrame:
    '''
    Read from Mongo Atlas.
        Args:
            uri (str): MongoDB connection URI.
            db_name (str): Database name.
            coll_name (str): Collection name.
            query (dict, optional): Query filter. Defaults to {}.
            limit (int, optional): Maximum number of documents to retrieve. Defaults to 100000.
    '''
    # Steps:
    #   1. Connect to MongoDB using MongoClient(uri)
    #   2. Select the database (db_name) and collection (coll_name)
    #   3. Fetch matching documents with coll.find(query, limit=limit), as a list
    #   4. Close the client connection
    #   5. If no documents were found, return an empty DataFrame
    #   6. Otherwise flatten the documents into a DataFrame with pd.json_normalize(docs)
    #      (this handles the nested "enrichment" sub-document) and return it
    # TODO: implement
    pass  


#### Exercise 2.3.2 : Load data from MongoDB Atlas

* In this exercise, you attempt to load enrichment documents from MongoDB Atlas using the helper function created earlier.
* After loading the data, you check its shape and preview a few rows to confirm that the enrichment dataset has been successfully ingested.

In [ ]:
# TODO:
#   1. Read MONGO_ATLAS_URI from the MONGO_ATLAS_URI environment variable (default to "")
#      -- never hardcode a connection string with a username/password in the notebook
#   2. Only attempt read_mongo_atlas(...) (db_name="workshop", coll_name="transactions") if a
#      URI is actually set; otherwise skip straight to the fallback
#   3. On any failure (missing URI, connection error, etc.), fall back to reading
#      DATA_DIR / "mongo.jsonl" with pd.read_json(..., lines=True, dtype=str) if it exists,
#      otherwise fall back to an empty DataFrame
#   4. Store the result in df_mongo_raw
#   5. Call inspect_dataframe(df_mongo_raw, "Mongo (Raw)", sample_col="eci_moto_cd")

##### ✅ Checkpoint: Data Acquisition
Run this before moving on — it confirms all three raw sources loaded correctly.

In [ ]:
try:
    assert 'df_settlement_raw' in globals() and not df_settlement_raw.empty, \
        "df_settlement_raw is missing or empty — check Exercise 2.1.2."
    assert 'df_api_raw' in globals(), "df_api_raw is missing — check Exercise 2.2.2."
    assert 'df_mongo_raw' in globals(), "df_mongo_raw is missing — check Exercise 2.3.2."
    assert "tran_id_raw" in df_settlement_raw.columns, \
        "df_settlement_raw is missing 'tran_id_raw' — check read_csv_strict / the settlement load."
    
    print(f"Checkpoint OK — settlement: {df_settlement_raw.shape}, "
          f"api: {df_api_raw.shape}, mongo: {df_mongo_raw.shape}")
except Exception as _e:
    print(f'\u26a0\ufe0f  Checkpoint skipped (not yet implemented): {_e}')


#### Summary

* Now you have successfully loaded transaction data from three distinct source types into three separate pandas DataFrames: `df_settlement_raw`, `df_api_raw`, and `df_mongo_raw`. 
* With these datasets now accessible as DataFrames, you are ready to begin exploring their contents and assessing data quality in the next activity.

## Activity 3 : Utility Function

* In this activity, you create small but important helper functions that standardize how placeholder values and missing entries are handled across all datasets.
* These utilities ensure that the settlement, API, and MongoDB sources follow consistent rules before any profiling, validation, or merging takes place.
Establishing these helpers early makes the downstream workflow cleaner, repeatable, and easier to maintain.

#### Exercise 3.1 : Normalization Helper

In this exercise, you define a utility function that prepares placeholder values for consistent normalization across your datasets.

In [ ]:
# Define the normalize_placeholders function

def normalize_placeholders(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean placeholder values in all object columns.
    This version is expanded for clarity and teaching.
    """
    # Steps:
    #   1. Work on a copy of df so the caller's DataFrame isn't mutated in place
    #   2. For every column with dtype "object":
    #        a. Cast to string and strip leading/trailing whitespace
    #        b. Replace any value that matches an entry in PLACEHOLDERS with pd.NA
    #   3. Return the cleaned DataFrame
    # TODO: implement
    pass 


#### Exercise 3.2 : Apply normalization to all the raw dataframes

In this exercise, you apply the normalization helper to each raw dataset to ensure consistent handling of placeholder values before further processing.

In [ ]:
# TODO: Call normalize_placeholders on each _raw DataFrame and store the results as
# df_settlement, df_api, df_mongo -- these become the working copies used for the rest
# of the notebook.

#### Check Your Normalized Data

In [ ]:
# TODO:
#   1. For every column in df_settlement_raw, count how many values are in PLACEHOLDERS
#      (hint: df_settlement_raw[col].isin(PLACEHOLDERS).sum())
#   2. Put the per-column counts into a Series/DataFrame sorted descending
#   3. Display the top 10 columns with the most placeholder values

In [ ]:
# TODO: Repeat the same counting logic as the previous cell, but on df_settlement
# (the normalized version) instead of df_settlement_raw. The counts should now be at or
# near zero, confirming normalize_placeholders worked.

##### ✅ Checkpoint: Normalization
Confirms placeholder tokens were actually replaced with missing values.

In [ ]:
try:
    _check_tokens = {"9999", "N/A", "NA", "None", "-", "0000"}
    for _name, _df in [("settlement", df_settlement), ("api", df_api), ("mongo", df_mongo)]:
        if _df.empty:
            continue
        _obj_cols = _df.select_dtypes(include="object").columns
        _leftover = {p for p in _check_tokens if (_df[_obj_cols] == p).any().any()}
        assert not _leftover, (
            f"{_name}: normalize_placeholders left raw placeholder tokens behind: {_leftover} "
            "— check Exercise 3.1/3.2."
        )
    
    print("Checkpoint OK — no raw placeholder tokens remain in df_settlement/df_api/df_mongo.")
except Exception as _e:
    print(f'\u26a0\ufe0f  Checkpoint skipped (not yet implemented): {_e}')


#### Summary

* You have now defined and applied utility functions that normalize placeholder values across all three datasets.
* With these foundational helpers in place, your data is better aligned and ready for structured profiling and quality assessment in the upcoming activities.

## Activity 4 : Preprocessing & flags

* In this activity, you establish a set of preprocessing steps and validity checks that help evaluate the quality of the raw datasets.
* You create utilities to extract standardized values, validate key identifiers, and generate basic data-quality flags such as PAN validity, transaction ID validity, and MCC consistency.
* These checks provide an initial view of how clean or inconsistent each dataset is before deeper profiling and transformation.

#### Exercise 4.1: Validating and cleaning Merchant Category Code (MCC) fields

In this exercise, you create a helper function that standardizes and extracts the base MCC code from labeled or placeholder values.

In [ ]:
# Define extract_code function
def extract_code(series: pd.Series) -> pd.Series:
    """
    Extracts the leading code from a labeled string field in the format "NNN - Label".
    - If the separator " - " is present, returns the code portion before it.
    - If not, returns the stripped original value.
    - Replaces any known placeholder values with pd.NA for consistency.
    Args:
        series (pd.Series): Input pandas Series containing labeled codes.
    Returns:
        pd.Series: Series of extracted codes or pd.NA where placeholders are found.
    """
    # Steps:
    #   1. Fill NA with "" and strip whitespace from every value
    #   2. Identify which rows contain the " - " separator
    #   3. For rows with the separator, split on " - " once and keep the left-hand part, stripped
    #   4. For rows without the separator (and not empty), keep the value as-is
    #   5. Replace any PLACEHOLDERS values with pd.NA
    #   6. Return the resulting Series
    # TODO: implement
    pass 


> 📘 **Primer:** Before writing `pan_valid_mask` and `tid_valid_mask` below, work through `SessionMaterials/regex_validation_sample.ipynb` — a short, self-contained notebook on regex-based validation with pandas `.str.match`.

#### Exercise 4.2 : Verify validity of PAN 

In this exercise, you create a Boolean mask that identifies which values follow the expected masked PAN format used in the dataset.

In [ ]:
# Define pan_valid_mask function
def pan_valid_mask(series: pd.Series) -> pd.Series:
    """
    Returns a Boolean mask indicating whether each value in the Series
    matches the masked PAN format used in the synthetic dataset:

        'XXXXXXXXXXXX1234'

    - First 12 characters must be 'X'
    - Last 4 characters must be digits (0-9)
    - Total length = 16 characters
    """
    # Steps:
    #   1. Fill NA with "" and cast to string
    #   2. Use a regex to check the value matches exactly 12 'X' characters followed by
    #      exactly 4 digits (str.match with na=False)
    #   3. Return the resulting Boolean Series
    # TODO: implement
    pass 


#### Exercise 4.3: Verify validity of TID

In this exercise, you create a Boolean mask to identify which values follow the required 15-digit numeric format for valid transaction IDs.
The Transaction ID (TID) is the primary key used to link Settlement, API, and Mongo datasets.

In [ ]:
# Define tid_valid_mask function

def tid_valid_mask(series: pd.Series) -> pd.Series:
    """
    Returns a Boolean mask indicating whether each value in the Series is a valid Transaction ID (TID).
    Validation rule for TID:
    - Must be exactly 15 characters long
    - Must contain only digits (0-9)
    """
    # Steps:
    #   1. Fill NA with "" and cast to string
    #   2. Use a regex to check the value is exactly 15 numeric digits (str.match with na=False)
    #   3. Return the resulting Boolean Series
    # TODO: implement
    pass 


#### Exercise 4.4 : Adding data quality flags 

In this exercise, you generate key data-quality flags for each dataset and produce a summary of basic validity metrics that will support later profiling and analysis

In [ ]:
# Define add_basic_flags function
def add_basic_flags(df: pd.DataFrame, name: str) -> Tuple[pd.DataFrame, dict]:
    """
    Add high-level data-quality flags to a dataset:
      - PAN validity
      - TID validity
      - MCC 4-digit code extraction + validity
      - row-level completeness score

    Also returns a summary dictionary with key dataset metrics.
    """
    # Steps:
    #   1. Work on a copy of df; add a "_source" column set to `name`
    #   2. "_pan_valid": pan_valid_mask(df["pymt_crd_acct_num"]) if that column exists, else False
    #   3. "_tid_valid": tid_valid_mask(df["tran_id_raw"]) if that column exists, else False
    #   4. If "mrch_catg_cd" exists: extract_code(...) into "mrch_catg_cd_code", then
    #      "_mcc_valid" = that code matches exactly 4 digits; otherwise set both to NA/False
    #   5. Build a summary dict: source, rows, pan_invalid, tid_invalid, mcc_invalid
    #      (counts of False in each flag), and median_completeness
    #      (df.notna().mean(axis=1).median())
    #   6. Return (df, summary)
    # TODO: implement
    pass 


#### Exercise 4.5 : Applying Data quality flags to each dataset

In this exercise, you apply the data-quality flagging function to each dataset and capture summary metrics that describe the state of the settlement, API, and MongoDB sources.

In [ ]:
# TODO:
#   1. Call add_basic_flags on df_settlement, df_api, and df_mongo (passing a name string
#      like "settlement"/"api"/"mongo"), reassigning each df and capturing its summary dict,
#      e.g. df_settlement, sum_set = add_basic_flags(df_settlement, "settlement")
#   2. Display the three summary dicts so you can compare data quality across sources

##### ✅ Checkpoint: Data-Quality Flags
Confirms add_basic_flags actually ran on all three datasets.

In [ ]:
try:
    for _name, _df in [("settlement", df_settlement), ("api", df_api), ("mongo", df_mongo)]:
        for _col in ("_pan_valid", "_tid_valid", "_mcc_valid", "_source"):
            assert _col in _df.columns, (
                f"{_name}: add_basic_flags did not create '{_col}' — check Exercise 4.4/4.5."
            )
    
    print("Checkpoint OK — quality flags present on df_settlement/df_api/df_mongo.")
except Exception as _e:
    print(f'\u26a0\ufe0f  Checkpoint skipped (not yet implemented): {_e}')


#### Summary

* You have now applied foundational data-quality checks across all three datasets.
* By generating flags for key fields and summarizing completeness and validity, you’ve created a clear picture of potential issues in the source data.
* This prepares you for the next stage of the workflow, where you will begin profiling the datasets in more detail and addressing data inconsistencies.

## Activity 5: Merge & canonicalization

* In this activity, you bring together the settlement, API, and MongoDB datasets into a single unified view.
* You perform a structured merge based on transaction IDs, resolve duplicate or conflicting fields, and create canonical columns using a consistent priority across sources.
* By consolidating the data into one standardized schema, you prepare a clean foundation for further analysis and validation in later steps.

* This is a crucial Activity where you will combine the three separate DataFrames into a single, unified dataset. 
* You will perform a series of left merges, using the settlement data as the authoritative source. 
* You will then use a `coalesce` function to intelligently combine overlapping columns from the different sources into one, following a defined order of precedence (settlement > API > mongo). 
* Finally, you will build the final `df_canonical` DataFrame that adheres to the target schema defined in the configuration.

> 📘 **Primer:** Before merging the three datasets below, work through `SessionMaterials/merge_sample.ipynb` — a short notebook on pandas `merge()`, join types, and suffix handling.

#### Exercise 5.1 : Merging Dataset 

In this exercise, you merge the settlement, API, and MongoDB datasets into a single table using tran_id_raw as the key, preserving the settlement records as the canonical base.

In [ ]:
# TODO:
#   1. Left-join df_settlement with df_api on "tran_id_raw", using suffixes ("_set", "_api"),
#      storing the result in `merged`
#   2. If "tran_id_raw" exists in df_mongo: first drop duplicate tran_id_raw values from
#      df_mongo (Mongo may contain repeats), then left-join the result into `merged` on
#      "tran_id_raw" with suffixes ("", "_mongo")
#   3. Print the shape of `merged` so you can confirm nothing unexpected happened

##### ✅ Checkpoint: Merge
Confirms the three-way merge ran and didn't drop settlement rows.

In [ ]:
try:
    assert 'merged' in globals(), "merged is missing — check Exercise 5.1."
    assert len(merged) >= len(df_settlement), (
        "merged has fewer rows than df_settlement — a left join on tran_id_raw "
        "should never drop settlement rows."
    )
    
    print(f"Checkpoint OK — merged shape: {merged.shape}")
except Exception as _e:
    print(f'\u26a0\ufe0f  Checkpoint skipped (not yet implemented): {_e}')


In [ ]:
# TODO: Display merged.head(3), and list all of merged's columns
# (merged.columns.tolist()) so you can see which source each field came from.

> 📘 **Primer:** Before writing `coalesce_columns_vectorized` below, work through `SessionMaterials/coalesce_sample.ipynb` — a short notebook on vectorized column-priority coalescing with `Series.where`.

#### Exercise 5.2: Function to coalesce columns

In this exercise, you define a helper function that combines the same field from settlement, API, and MongoDB sources into a single canonical column using a consistent priority order.

In [ ]:
# Define coalesce_columns_vectorized function

def coalesce_columns_vectorized(df: pd.DataFrame, base_name: str) -> pd.Series:
    """
    Combine the same field coming from 3 different datasets (Settlement, API, Mongo) into one canonical column using the priority:
        Settlement > API > Mongo

    Example:
        Input columns in merged DF:
            tran_curr_cd_set   <- from settlement
            tran_curr_cd_api   <- from API
            tran_curr_cd        <- from mongo

        Output:
            a single Series where:
                - if settlement has a value, use it
                - else if API has a value, use it
                - else if Mongo has a value, use it
                - else NA

    This is vectorized (uses pandas operations, no loops), so it's very fast even for thousands of rows.
    """
    # Steps:
    #   1. Work out the expected column names: f"{base_name}_set", f"{base_name}_api",
    #      and the plain base_name (for Mongo)
    #   2. Start with an all-NA Series aligned to df's index
    #   3. If the settlement column exists, fill from it wherever the running result is
    #      still NA (Series.where)
    #   4. Do the same for the API column, then the Mongo column, always only filling
    #      positions that are still NA
    #   5. Return the final coalesced Series
    # TODO: implement
    pass  


#### Exercise 5.3 : Build canonical DataFrame using coalescing

In this exercise, you build the canonical dataset by iterating through the predefined schema and coalescing each field from the merged sources into a single unified column.

In [ ]:
# TODO:
#   1. Create an empty DataFrame df_canonical with the same index as `merged`
#   2. For each column name in CANONICAL_COLS:
#        a. Try to build it via coalesce_columns_vectorized(merged, col)
#        b. If that raises an exception, fall back to merged.get(col, <all-NA Series>)

##### ✅ Checkpoint: Canonical Build
Confirms every canonical column got built and no rows were lost while coalescing.

In [ ]:
try:
    _missing_cols = [c for c in CANONICAL_COLS if c not in df_canonical.columns]
    assert not _missing_cols, (
        f"df_canonical is missing canonical columns: {_missing_cols} — check Exercise 5.3's coalescing loop."
    )
    assert len(df_canonical) == len(merged), (
        "df_canonical row count does not match merged — coalescing should not change row count."
    )
    
    print(f"Checkpoint OK — df_canonical has all {len(CANONICAL_COLS)} canonical columns, {len(df_canonical)} rows.")
except Exception as _e:
    print(f'\u26a0\ufe0f  Checkpoint skipped (not yet implemented): {_e}')


#### Exercise 5.4 : Adding Audit flags

In this exercise, you generate key validation flags for the canonical dataset—covering PAN format, transaction ID format, MCC structure, and overall row completeness—to support downstream quality assessment.

In [ ]:
# TODO:
#   1. is_pan_valid: True where pymt_crd_acct_num matches the masked PAN format
#      (reuse pan_valid_mask); default to False if the column is missing
#   2. is_tid_valid: True where tran_id_raw is exactly 15 digits (reuse tid_valid_mask);
#      default to False if the column is missing
#   3. is_mcc_valid: define a small helper that returns True when a value matches
#      "NNNN - Label" (4 digits, " - ", label), then apply it to mrch_catg_cd;
#      default to False if the column is missing
#   4. _completeness: the fraction of CANONICAL_COLS that are non-null, per row
#      (df_canonical[CANONICAL_COLS].notna().sum(axis=1) / len(CANONICAL_COLS))

#### Exercise 5.5 : Concise Summary of Merge and Canonicalization

In this exercise, you generate a summary of key validation and completeness metrics for the canonical dataset, giving a clear overview of data quality after merging and standardization.

In [ ]:
# TODO: Build a summary dict with merged_rows, canonical_rows, and the valid/invalid
# counts for is_pan_valid / is_tid_valid / is_mcc_valid, then display it.

#### Exercise 5.6: Display a short canonical sample 

In this exercise, you review a sample of the canonicalized dataset by displaying key fields and validation flags, along with its overall shape, to confirm that the merged and standardized output looks as expected.

In [ ]:
# TODO:
#   1. Pick a handful of columns to show (e.g. tran_id_raw, pymt_crd_acct_num, mrch_catg_cd,
#      tran_curr_cd, is_pan_valid, is_mcc_valid, is_tid_valid, _completeness) -- keep only the
#      ones that actually exist in df_canonical
#   2. Display the first 8 rows of just those columns, plus df_canonical's overall shape

#### Summary

* You have now merged all three datasets and transformed them into a canonical structure.
* By coalescing fields, generating validation flags, and reviewing completeness metrics, you’ve created a harmonized dataset that brings together all available transaction information.
* This unified view provides the clarity and consistency needed for deeper profiling, enrichment, and downstream analytics.

## Activity 6: Cleaning & canonical fixes

* In this Activity, you will perform the final, detailed cleaning operations on your merged `df_canonical` DataFrame. 
* You will remove duplicate transactions, nullify any remaining invalid PANs, normalize Merchant Category Codes (MCCs) into a standard format, and impute (fill in) missing values for a few low-risk categorical fields using the mode. 
* You will also perform a quality analysis by calculating the top missing fields before generating a detailed `ydata-profiling` report.

#### Exercise 6.1 : Spotting Missing values 

In this exercise, you define a helper function that reports which columns contain the most missing values, giving you a quick overview of data gaps that may need attention.

In [ ]:
# Develop utility to inspect missingness
def top_missing(df: pd.DataFrame, top_n: int = 12) -> pd.DataFrame:
    """
    Return a sorted table of the columns with the most missing values.
    Parameters:
        df (pd.DataFrame): The dataframe to evaluate.
        top_n (int): How many top missing-value columns to return.
    Returns:
        pd.DataFrame: A table with:
            - 'missing'      : number of missing values per column
            - 'pct_missing'  : missing values as percentage of total rows
    """
    # Steps:
    #   1. Count missing values per column (df.isna().sum())
    #   2. Sort descending and keep the top `top_n`
    #   3. Build a DataFrame with an absolute "missing" count and a "pct_missing"
    #      (missing count divided by len(df))
    #   4. Return that DataFrame
    # TODO: implement
    pass 


#### Exercise 6.2 : Display the Summary of the Canonical Dataset

In this exercise, you generate a missing-value summary for the cleaned dataset to identify which columns have the highest proportion of gaps.

In [ ]:
# TODO: Call top_missing(df_canonical) and display the result.

#### Exercise 6.3 : Profile Reporting

In this exercise, you generate a profiling report for a sample of the canonical dataset and save it as an HTML file to review key data patterns before applying further cleaning steps.

In [ ]:
class ProfileReport:
    """Minimal drop-in that generates a readable HTML data-profile report."""

    def __init__(self, df, title="Profile Report", minimal=True):
        self._df = df
        self._title = title

    @staticmethod
    def _fig_to_b64(fig):
        buf = io.BytesIO()
        fig.savefig(buf, format="png", bbox_inches="tight")
        buf.seek(0)
        return base64.b64encode(buf.read()).decode()

    def _overview_section(self):
        df = self._df
        rows, cols = df.shape
        missing_cells = int(df.isna().sum().sum())
        total_cells = rows * cols
        dup_rows = int(df.duplicated().sum())
        items = [
            ("Number of rows", f"{rows:,}"),
            ("Number of columns", f"{cols:,}"),
            ("Total missing cells", f"{missing_cells:,}"),
            ("Missing cells (%)", f"{100*missing_cells/total_cells:.1f}%"),
            ("Duplicate rows", f"{dup_rows:,}"),
            ("Memory usage", f"{df.memory_usage(deep=True).sum()/1024:.1f} KB"),
        ]
        rows_html = "".join(f"<tr><td>{k}</td><td><b>{v}</b></td></tr>" for k, v in items)
        return f"<h2>Overview</h2><table border='1' cellpadding='4'>{rows_html}</table>"

    def _missing_bar(self):
        df = self._df
        miss = df.isna().mean().sort_values(ascending=False)
        miss = miss[miss > 0].head(20)
        if miss.empty:
            return "<p>No missing values found.</p>"
        fig, ax = plt.subplots(figsize=(10, max(3, len(miss) * 0.35)))
        miss.plot.barh(ax=ax, color="steelblue")
        ax.set_xlabel("Fraction missing")
        ax.set_title("Top columns by missing-value fraction")
        ax.invert_yaxis()
        plt.tight_layout()
        b64 = self._fig_to_b64(fig)
        plt.close(fig)
        return f"<h2>Missing Values</h2><img src='data:image/png;base64,{b64}'/>"

    def _column_stats(self):
        df = self._df
        parts = ["<h2>Column Statistics</h2>"]
        for col in df.columns:
            s = df[col]
            n_miss = int(s.isna().sum())
            pct_miss = 100 * n_miss / len(s) if len(s) else 0
            n_unique = s.nunique()
            info = (f"<b>{_html.escape(col)}</b> — dtype: {s.dtype} | "
                    f"missing: {n_miss} ({pct_miss:.1f}%) | unique: {n_unique}")
            top_html = ""
            if s.dtype == object or str(s.dtype).startswith("string"):
                top = s.value_counts().head(5)
                if not top.empty:
                    rows_h = "".join(f"<tr><td>{_html.escape(str(k))}</td><td>{v}</td></tr>"
                                     for k, v in top.items())
                    top_html = (f"<table border='1' cellpadding='3' style='margin-left:20px'>"
                                f"<tr><th>Value</th><th>Count</th></tr>{rows_h}</table>")
            else:
                try:
                    desc = s.describe()
                    rows_h = "".join(f"<tr><td>{k}</td><td>{v:.4g}</td></tr>"
                                     for k, v in desc.items())
                    top_html = (f"<table border='1' cellpadding='3' style='margin-left:20px'>"
                                f"{rows_h}</table>")
                except Exception:
                    pass
            parts.append(f"<div style='margin-bottom:12px'>{info}{top_html}</div>")
        return "\n".join(parts)

    def to_file(self, output_path):
        overview = self._overview_section()
        missing  = self._missing_bar()
        col_stats = self._column_stats()
        html_doc = f"""<!DOCTYPE html>
<html><head><meta charset='utf-8'/>
<title>{_html.escape(self._title)}</title>
<style>body{{font-family:Arial,sans-serif;margin:24px}}h1{{color:#2c3e50}}
h2{{color:#34495e;border-bottom:1px solid #ccc;padding-bottom:4px}}
table{{border-collapse:collapse;font-size:13px}}td,th{{padding:4px 8px}}</style>
</head><body><h1>{_html.escape(self._title)}</h1>
{overview}{missing}{col_stats}</body></html>"""
        with open(output_path, "w", encoding="utf-8") as fh:
            fh.write(html_doc)
        print(f"Profile saved \u2192 {output_path}")

try:
    from pymongo import MongoClient
except ImportError:
    MongoClient = None
    print("pymongo not available — MongoDB steps will use the local JSONL fallback.")

In [ ]:
# TODO:
#   1. Take a manageable sample of df_canonical (e.g. up to 2000 rows) to keep profiling fast
#   2. Build a ProfileReport over that sample (minimal=True speeds this up)
#   3. Save it with profile.to_file(OUTPUT_DIR / "profile_before.html")


> 📘 **Primer:** Before deduplicating below, work through `SessionMaterials/deduplication_sample.ipynb` — a short notebook on `duplicated()`/`drop_duplicates()` and auditing what gets dropped.

#### Exercise 6.4: Cleaning Data : Deduplicate by tran_id_raw

In this exercise, you identify and remove duplicate transaction IDs from the cleaned dataset while capturing the dropped records for reference, helping ensure uniqueness and consistency in further analysis.

In [ ]:
# TODO:
#   1. Record the row count before dedup (before_rows = len(df_canonical))
#   2. Build a Boolean mask of duplicate tran_id_raw values, keeping the first occurrence
#      (df_canonical["tran_id_raw"].duplicated(keep="first"))
#   3. Save the duplicate rows separately (e.g. dropped_dupes) for auditing
#   4. Keep only the non-duplicate rows and reset the index
#   5. Print how many rows were dropped and how many remain

##### ✅ Checkpoint: Deduplication
Confirms no duplicate transaction IDs remain after Exercise 6.4.

In [ ]:
try:
    assert df_canonical["tran_id_raw"].duplicated().sum() == 0, (
        "df_canonical still has duplicate tran_id_raw values — check Exercise 6.4's dedup logic."
    )
    
    print(f"Checkpoint OK — no duplicate tran_id_raw remain ({len(df_canonical)} rows).")
except Exception as _e:
    print(f'\u26a0\ufe0f  Checkpoint skipped (not yet implemented): {_e}')


#### Helper Exercise: A reusable normalization-reporting function
The next three cleaning steps (PAN, MCC, currency) each end the same way: report how many values are now valid, how many were nullified/unmapped, and show the most common resulting values. Define that reporting logic once here and reuse it for all three.

In [ ]:
def report_normalization(df: pd.DataFrame, col: str, valid_flag: str = None, top_n: int = 10) -> None:
    """
    Print a quick, consistent before/after summary for a normalized column: total rows,
    how many are now valid/non-null, how many are still missing/invalid, and the top-N
    most common resulting values.

    Args:
        df (pd.DataFrame): The DataFrame containing the normalized column.
        col (str): The column that was just normalized/cleaned.
        valid_flag (str, optional): A companion boolean flag column (e.g. "is_pan_valid")
            to report valid/invalid counts from. If omitted, validity is inferred from
            non-null values in `col`.
        top_n (int): How many of the most common resulting values to display.
    """
    # Steps:
    #   1. total = len(df)
    #   2. If valid_flag is given and present in df: valid = df[valid_flag].sum(),
    #      print total/valid/invalid using that flag
    #   3. Otherwise: mapped = df[col].notna().sum(), print total/mapped/unmapped
    #   4. Print a header, then display(df[col].value_counts().head(top_n))
    # TODO: implement
    pass 


#### Exercise 6.5.1 : PAN Handling

In this exercise, you preserve the PAN validity flag while clearing out any PAN values that fail the format check, ensuring that downstream steps do not rely on invalid identifiers.

In [ ]:
# TODO: Where is_pan_valid is False, set pymt_crd_acct_num to pd.NA (nullify it) --
# but leave the is_pan_valid flag itself untouched so the invalidity is still visible
# for auditing.

#### Exercise 6.5.2 : Modified PAN Data Inspection

Call your `report_normalization` helper to confirm the PAN cleaning worked as expected.

In [ ]:
# TODO: report_normalization(df_canonical, "pymt_crd_acct_num", valid_flag="is_pan_valid")

#### Exercise 6.6.1 : Merchant Category Code (MCC) Normalization

In this exercise, you normalize MCC values into a consistent labeled format and update the MCC validity flag based on the cleaned results.

In [ ]:
# Develop normalize_mcc_val
def normalize_mcc_val(val: Any) -> Any:
    """
    Normalize a merchant category code (MCC) value into the canonical
    "NNNN - Label" form where possible.
    - If already in "NNNN - Label" format, return it unchanged.
    - If a bare 4-digit code is found at the start, map it to a known label
      if one exists in the reference mapping.
    - Otherwise, return pd.NA.
    """
    # Steps:
    #   1. If val is missing (pd.isna), return pd.NA
    #   2. Strip the value to a string; if it already matches "NNNN - " at the start, return as-is
    #   3. Otherwise, try to extract a leading 4-digit code
    #   4. If found, look it up in a small dict of known MCC codes -> labels
    #      (e.g. "5411": "5411 - Grocery Stores", "5311": "5311 - Department Stores",
    #       "5542": "5542 - Automated Fuel Dispensers", "5812": "5812 - Restaurants",
    #       "6011": "6011 - ATM Cash", "5992": "5992 - Florists"); return the mapped label
    #      if found, else pd.NA
    #   5. If no 4-digit code was found at all, return pd.NA
    # TODO: implement
    pass 

# TODO: Apply normalize_mcc_val to df_canonical["mrch_catg_cd"], then update the
# "is_mcc_valid" flag to reflect which values are non-null afterward.

#### Exercise 6.6.2 : Inspect MCC normalization summary

Call your `report_normalization` helper to confirm the MCC normalization worked as expected.

In [ ]:
# TODO: report_normalization(df_canonical, "mrch_catg_cd", valid_flag="is_mcc_valid")

#### Exercise 6.7.1 : Currency reconciliation

In this exercise, you normalize the transaction currency into a clean three-digit code and create a corresponding validity flag to support later auditing.

In [ ]:
# Define normalize_currency_code function

def normalize_currency_code(val):
    """
    Normalize a transaction currency code into a clean 3-digit numeric string.
    - If already in "NNN - Label" format, extract and validate the leading code.
    - If a bare 3-digit code, return it as-is.
    - Otherwise, return pd.NA.
    """
    # Steps:
    #   1. If val is missing (pd.isna), return pd.NA
    #   2. Strip the value to a string
    #   3. If it contains " - ", split on it and take the left-hand part as the candidate code;
    #      keep it only if it's exactly 3 digits, else return pd.NA
    #   4. If there's no " - ", check whether the whole value is already exactly 3 digits;
    #      if so return it, otherwise return pd.NA
    # TODO: implement
    pass 

# TODO: Apply normalize_currency_code to df_canonical["tran_curr_cd"], store the
# result in "_tran_curr_code", and set "_tran_curr_valid" to True wherever that
# result is non-null.

#### Exercise 6.7.2 : Currency Code Normalization Summary

Call your `report_normalization` helper to confirm the currency normalization worked as expected.

In [ ]:
# TODO: report_normalization(df_canonical, "_tran_curr_code", valid_flag="_tran_curr_valid")

#### Exercise 6.8 : Imputation of low risk categorical fields with Mode

In this exercise, you impute missing values for selected categorical fields by filling them with their most frequent (mode) value to improve completeness in low-risk attributes.

In [ ]:
# TODO:
#   1. For each of these low-risk categorical columns: pos_entry_mode_cd, pos_env_cd,
#      pos_terml_cpbty_cd, pin_entry_cpbty_cd
#   2. If the column exists in df_canonical, compute its mode (most frequent non-null value)
#   3. Fill missing values in that column with the mode value
#   4. If computing the mode fails for any reason, fall back to filling with "UNKNOWN"

#### Exercise 6.9 : Final summary of the cleaning operation

In this exercise, you generate a final cleaning summary capturing row counts, duplicates removed, and the remaining invalid flags across key fields to confirm the overall impact of your cleaning steps.

In [ ]:
# TODO: Build and display a clean_summary dict with: rows_before, rows_after,
# dropped_duplicates, and the final invalid counts for PAN/MCC/TID/currency
# (each only if that flag column exists in df_canonical).

#### Exercise 6.10

In this activity display the data in the selected columns from the cleaned dataset, allowing us to observe the impact of the data cleaning steps.

In [ ]:
# TODO: Display the first 10 rows of a handful of key columns (tran_id_raw,
# pymt_crd_acct_num, is_pan_valid, mrch_catg_cd, is_mcc_valid, _tran_curr_code,
# _tran_curr_valid, is_tid_valid, _completeness_after) -- keep only the ones that exist.

#### Summary

* You now have a fully cleaned and production-ready dataset. 
* You have also practiced key data cleaning techniques like deduplication, normalization, and imputation. 
* You have a summary report (`profile_before.html`) and printed metrics that quantify the quality of the data before the cleaning pipeline was run.

## Activity 7: Visualizations

* In the Activity, we will inspect a sample of the fully cleaned data to visually confirm the results. 
* You will then create several visualizations to compare the state of the data "before" and "after" the cleaning process, focusing on missingness and data validity.

#### Exercise 7.1 : Visualize before/after missingness and simple data-quality metrics.

In this exercise, you compare and visualize column-level missingness before and after cleaning, producing a side-by-side view and a bar chart to inspect the impact of your cleaning steps.

In [ ]:
# TODO:
#   1. For the CANONICAL_COLS that exist in both `merged` (before) and df_canonical (after),
#      compute the fraction of missing values per column; keep the top 12 by "before"
#      missingness
#   2. Combine both into one DataFrame with columns "before" and "after"
#   3. Plot it as a horizontal bar chart (DataFrame.plot.barh) to compare before vs after

#### Exercise 7.2 : PAN Validity Visualization

In this actiivty visualize the number of valid and invalid PANs in the cleaned canonical dataset.

In [ ]:
# TODO:
#   1. If is_pan_valid exists in df_canonical, count how many rows are valid vs invalid
#   2. Plot a simple bar chart with two bars: "valid" and "invalid"
#   3. If the column is missing, print a warning instead of plotting

#### Summary

* In this activity, you visualized how the cleaning steps affected data completeness by comparing missingness before and after cleaning.
* Through a simple side-by-side chart, you gained a clear view of which fields improved the most and where gaps still remain.
* This helps validate the effectiveness of your preprocessing work and highlights any areas that may need further attention.

## Activity 8: Persistence and metadata

* In this activity, you prepare the cleaned canonical dataset for reuse by saving it in multiple output formats.
* You create durable artifacts—Parquet, SQLite, and metadata JSON—that can be consumed by analytics tools, machine-learning workflows, or downstream services.
* This step ensures that the results of your cleaning and standardization process are easily accessible, queryable, and traceable.

#### Exercise 8.1 : Defining the path for persisting the dataset

In this exercise, you define the output locations for saving the cleaned dataset in multiple formats—Parquet, SQLite, and metadata JSON—so the results can be reused in downstream workflows.

In [ ]:
# TODO: Define PARQUET_OUT, SQLITE_OUT, and META_OUT as paths under OUTPUT_DIR, e.g.
# "reconstructed_clean.parquet", "reconstructed_clean.db", "ingest_metadata.json"

#### Exercise 8.2 : Parquet File

In this exercise, you save the cleaned canonical dataset as a Parquet file, providing a fast, efficient columnar format for downstream use.

In [ ]:
# TODO: Save df_canonical to PARQUET_OUT using DataFrame.to_parquet(..., index=False)

#### Exercise 8.3 : SQLite

In this exercise, you store the cleaned canonical dataset in a SQLite database, making it accessible through SQL queries and convenient for lightweight analytics or integration tasks.

In [ ]:
# TODO:
#   1. Open a sqlite3 connection to SQLITE_OUT
#   2. Write df_canonical to a table named "clean_transactions" using
#      DataFrame.to_sql(..., if_exists="replace", index=False)

#### Exercise 8.4 : Metadata information

In this exercise, you generate and save ingestion metadata—capturing final row counts and key validity metrics—both in SQLite and as a standalone JSON file, providing useful audit information for downstream processes.

In [ ]:
# TODO:
#   1. Build a metadata dict: rows_final, saved_at (current UTC timestamp), and the final
#      invalid counts for PAN/MCC/TID (each only if that flag column exists)
#   2. Write that dict to the SQLite connection as an "ingest_metadata" table, then close
#      the connection
#   3. Also write the same dict to META_OUT as a JSON file

##### ✅ Checkpoint: Persistence
Confirms all three output files were written and their row counts agree with df_canonical.

In [ ]:
try:
    assert PARQUET_OUT.exists(), f"Parquet file not found at {PARQUET_OUT} — check Exercise 8.2."
    assert SQLITE_OUT.exists(), f"SQLite file not found at {SQLITE_OUT} — check Exercise 8.3."
    assert META_OUT.exists(), f"Metadata JSON not found at {META_OUT} — check Exercise 8.4."
    
    _parquet_rows = len(pd.read_parquet(PARQUET_OUT))
    assert _parquet_rows == len(df_canonical), (
        f"Parquet row count ({_parquet_rows}) does not match df_canonical ({len(df_canonical)})."
    )
    
    _conn = sqlite3.connect(SQLITE_OUT)
    _sqlite_rows = pd.read_sql("SELECT COUNT(*) AS n FROM clean_transactions", _conn)["n"].iloc[0]
    _conn.close()
    assert _sqlite_rows == len(df_canonical), (
        f"SQLite row count ({_sqlite_rows}) does not match df_canonical ({len(df_canonical)})."
    )
    
    print("Checkpoint OK — Parquet, SQLite, and metadata JSON all saved and row counts match df_canonical.")
except Exception as _e:
    print(f'\u26a0\ufe0f  Checkpoint skipped (not yet implemented): {_e}')


#### Summary

* You have a complete, cleaned, and persisted dataset ready for downstream use. 
* You have created meaningful visualizations that demonstrate the value of your data quality work, 
* You have also experience saving your results in multiple formats. 
* You have now completed a full, end-to-end data preparation workflow.

## Conclusion

You have now completed the full end-to-end workflow for preparing real-world transactional data for analytics and AI applications.
Across this workshop, you worked with data from multiple systems, explored common quality issues, applied structured cleaning techniques, and unified the results into a reliable canonical dataset.

By the end of these activities, you accomplished the following:
* Loaded and inspected datasets from CSV, API, and MongoDB sources
* Standardized placeholder handling and applied key validation rules
* Merged disparate data structures into a consistent, well-defined schema
* Identified and corrected issues such as invalid PANs, MCC inconsistencies, missing currency codes, and duplicated transaction IDs
* Generated data-quality summaries and visualizations to validate each cleaning step
* Produced a final, analysis-ready dataset and persisted it in multiple formats for downstream use

You now have a solid foundation in the essential techniques used across data engineering, analytics, and machine-learning workflows.
The canonical dataset you built can be further explored, modeled, enriched, or integrated into larger systems.